In [5]:
import wrds
import pandas as pd
import os
import time     
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time

In [6]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [7]:
query = """
SELECT DISTINCT
    c.cik,
    c.conm,
    l.lpermno AS permno,
    m.start,
    m.ending
FROM crsp.msp500list m

JOIN crsp.ccmxpf_linktable l
    ON m.permno = l.lpermno
    AND l.linktype IN ('LU', 'LC')
    AND l.linkprim IN ('P', 'C')

JOIN comp.company c
    ON l.gvkey = c.gvkey

WHERE c.cik IS NOT NULL
    AND m.start <= '2025-12-31'
    AND (m.ending IS NULL OR m.ending >= '2000-01-01')
"""

df = db.raw_sql(query)

bad_keywords = [
    "FUND", "ETF", "TRUST", "PORTFOLIO",
    "SERIES", "INVESTMENT", "SICAV"
]

df = df[~df['conm'].str.upper().str.contains('|'.join(bad_keywords), na=False)]

ciks = df['cik'].dropna().astype(str).str.zfill(10).unique()

cik_df = pd.DataFrame({
    "cik": ciks
})

cik_df.to_csv("sp500_historical_ciks.csv", index=False)

print(f"Saved {len(ciks)} CIKs")

Saved 1046 CIKs
